# Train and evaluate MosquitoFusion with YOLOv8n

Run this notebook in Google Colab with a GPU runtime. It downloads the MosquitoFusion dataset from Kaggle, checks the YOLO annotations, trains a pretrained YOLOv8n model for 30 epochs, evaluates on the dataset's test split, and packages reproducibility details and prediction images.

Kaggle lists the dataset license as MIT; review its terms before use or redistribution. The authors' paper reports mAP@50 0.571, precision 0.734, and recall 0.505. Those are paper results, **not** this notebook's results. Only use the metrics printed by the evaluation cell as your run's results.

Do not put Kaggle credentials in the notebook or Git. Upload `kaggle.json` only into this temporary Colab session when prompted. Colab storage is temporary; download the final archive before the runtime ends.

In [ ]:
# Select Runtime > Change runtime type > T4 GPU (or another available GPU) before continuing.
from pathlib import Path
import os
import subprocess
import sys

PROJECT = Path('/content/mosquito-detection-yolo')
if not PROJECT.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/Kadimane/mosquito-detection-yolo.git', str(PROJECT)],
        check=True,
    )
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is available. In Colab, select Runtime > Change runtime type > GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))


## Kaggle access

Accept the dataset's terms on Kaggle first. Create a Kaggle API token from your Kaggle account settings, then upload the downloaded `kaggle.json` below. The file is stored with restrictive permissions in this Colab runtime only; never commit it or paste its contents into a notebook cell.

In [ ]:
from google.colab import files

uploaded = files.upload()
if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('Upload the Kaggle API token file named kaggle.json.')
kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(mode=0o700, parents=True, exist_ok=True)
token_path = kaggle_dir / 'kaggle.json'
token_path.write_bytes(uploaded['kaggle.json'])
token_path.chmod(0o600)
print('Kaggle credentials configured for this Colab session.')


In [ ]:
DATASET_SLUG = 'faiyazabdullah/mosquitofusion-dataset'
DATA_ROOT = Path('/content/mosquitofusion-data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', DATASET_SLUG, '-p', str(DATA_ROOT), '--unzip'],
    check=True,
)

yaml_files = sorted(DATA_ROOT.rglob('data.yaml'))
if len(yaml_files) != 1:
    raise RuntimeError(f'Expected exactly one data.yaml in {DATA_ROOT}, found: {yaml_files}')
DATA_YAML = yaml_files[0]
print('Dataset config:', DATA_YAML)


In [ ]:
import yaml
from src.mosquito_detector.data.validate import validate_yolo_dataset
from src.mosquito_detector.train import load_dataset_config

with DATA_YAML.open(encoding='utf-8') as stream:
    dataset_yaml = yaml.safe_load(stream)
if not isinstance(dataset_yaml, dict):
    raise ValueError(f'Expected a YAML mapping in {DATA_YAML}')

data_root = Path(dataset_yaml.get('path', '.')).expanduser()
if not data_root.is_absolute():
    data_root = (DATA_YAML.parent / data_root).resolve()
else:
    data_root = data_root.resolve()

def resolve_split(split):
    if split not in dataset_yaml or not dataset_yaml[split]:
        raise ValueError(f'The dataset config must define a {split!r} split.')
    values = dataset_yaml[split] if isinstance(dataset_yaml[split], list) else [dataset_yaml[split]]
    resolved = []
    aliases = {'train': ('train',), 'val': ('val', 'valid'), 'test': ('test',)}[split]
    for value in values:
        candidate = Path(value).expanduser()
        if not candidate.is_absolute():
            candidate = data_root / candidate
        if not candidate.is_dir():
            discovered = set()
            for alias in aliases:
                discovered.update(DATA_ROOT.rglob(f'{alias}/images'))
                discovered.update(DATA_ROOT.rglob(f'images/{alias}'))
            discovered = {path.resolve() for path in discovered if path.is_dir()}
            if len(discovered) != 1:
                raise FileNotFoundError(f'Could not uniquely resolve {split} image path {value!r}; candidates: {sorted(map(str, discovered))}')
            candidate = discovered.pop()
        resolved.append(str(candidate.resolve()))
    return resolved if isinstance(dataset_yaml[split], list) else resolved[0]

for split in ('train', 'val', 'test'):
    dataset_yaml[split] = resolve_split(split)
dataset_yaml['path'] = str(data_root)
RUNTIME_DATA_YAML = Path('/content/mosquitofusion-data.yaml')
RUNTIME_DATA_YAML.write_text(yaml.safe_dump(dataset_yaml, sort_keys=False), encoding='utf-8')
dataset = load_dataset_config(RUNTIME_DATA_YAML)
class_count = len(dataset.names)

def label_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    image_indexes = [index for index, part in enumerate(parts) if part.lower() == 'images']
    if not image_indexes:
        raise ValueError(f'Expected an images directory in the YOLO path: {images_dir}')
    parts[image_indexes[-1]] = 'labels'
    return Path(*parts)

split_counts = {}
for split in ('train', 'val', 'test'):
    image_dirs = getattr(dataset, split)
    image_dirs = image_dirs if isinstance(image_dirs, list) else [image_dirs]
    split_reports = []
    for images_dir in image_dirs:
        report = validate_yolo_dataset(images_dir, label_dir_for(images_dir), class_count)
        print(f'[{split}] {report.summary()}')
        if not report.is_valid:
            raise ValueError(f'Invalid {split} split; fix the listed image/label problems before training.')
        split_reports.append(report)
    split_counts[split] = sum(report.image_count for report in split_reports)
print('Class names:', dataset.names)
print('Image counts:', split_counts)


## Train

This uses pretrained YOLOv8n weights, 30 epochs, 640px images, batch size 8, and Colab GPU 0. It writes to `/content/outputs`; it does not change `configs/train.yaml` or the downloaded dataset.

In [ ]:
from datetime import datetime, timezone
import time
from src.mosquito_detector.train import train_model

TRAIN_CONFIG = Path('/content/mosquito-colab-train.yaml')
training_options = {
    'model_weights': 'yolov8n.pt',
    'epochs': 30,
    'image_size': 640,
    'batch_size': 8,
    'device': '0',
    'output_dir': '/content/outputs/runs',
    'run_name': 'mosquitofusion-yolov8n',
    'data_config': str(RUNTIME_DATA_YAML),
    'exist_ok': True,
}
TRAIN_CONFIG.write_text(yaml.safe_dump(training_options, sort_keys=False), encoding='utf-8')
training_started = datetime.now(timezone.utc)
started_clock = time.monotonic()
training_result = train_model(config_path=TRAIN_CONFIG)
training_duration_seconds = time.monotonic() - started_clock
best_weights = training_result.output_dir / 'weights' / 'best.pt'
if not best_weights.is_file():
    raise FileNotFoundError(f'Training did not produce the expected checkpoint: {best_weights}')
print('Best weights:', best_weights)
print(f'Training duration: {training_duration_seconds / 60:.1f} minutes')


## Evaluate on the held-out test split and make examples

The metrics below are measured by this run. The inference cell saves annotated copies of test images; it never edits source images.

In [ ]:
import json
import random
import shutil
import zipfile
from src.mosquito_detector.evaluate import evaluate_model
from src.mosquito_detector.inference import run_inference

RESULTS_DIR = Path('/content/mosquito-results')
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
evaluation = evaluate_model(
    weights=best_weights,
    dataset_config_path=RUNTIME_DATA_YAML,
    output_dir=Path('/content/outputs/plots'),
    split='test',
    image_size=640,
    batch_size=8,
    device='0',
)
metrics = evaluation.metrics.as_dict()
print('Measured test metrics:', json.dumps(metrics, indent=2))
shutil.copy2(evaluation.plot_path, RESULTS_DIR / 'metrics.png')

test_image_dirs = dataset.test if isinstance(dataset.test, list) else [dataset.test]
inference_results = []
for test_image_dir in test_image_dirs:
    inference_results.extend(run_inference(
        weights=best_weights,
        source=test_image_dir,
        output_dir=PREDICTIONS_DIR,
        confidence=0.25,
        image_size=640,
        device='0',
    ))
with_detections = [result for result in inference_results if result.detections]
sample_pool = with_detections if len(with_detections) >= 3 else list(inference_results)
sample_count = min(3, len(sample_pool))
selected = random.Random(42).sample(sample_pool, sample_count)
for index, image_result in enumerate(selected, start=1):
    if image_result.annotated_image is None:
        raise RuntimeError(f'No annotated output was saved for {image_result.source}')
    shutil.copy2(image_result.annotated_image, RESULTS_DIR / f'sample_{index}.jpg')
    print(f"sample_{index}.jpg: {len(image_result.detections)} detection(s)")
if sample_count < 3:
    print(f'Only {sample_count} test image(s) were available for the sample gallery.')

summary = {
    'dataset': 'MosquitoFusion',
    'dataset_source': 'https://www.kaggle.com/datasets/faiyazabdullah/mosquitofusion-dataset',
    'dataset_version': 3,
    'dataset_license_as_listed_by_kaggle': 'MIT (review source terms before redistribution)',
    'dataset_counts': split_counts,
    'class_names': dataset.names,
    'model': 'YOLOv8n (pretrained yolov8n.pt)',
    'epochs': 30,
    'image_size': 640,
    'batch_size': 8,
    'evaluation_split': evaluation.split,
    'metrics': metrics,
    'training_started_utc': training_started.isoformat(),
    'training_duration_seconds': round(training_duration_seconds, 2),
    'weights_file': 'best.pt',
}
(RESULTS_DIR / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')

archive_path = Path('/content/mosquito-results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS_DIR.parent))
    archive.write(best_weights, 'mosquito-results/best.pt')
print('Results archive:', archive_path)


In [ ]:
from google.colab import files

files.download('/content/mosquito-results.zip')


After downloading and reviewing `summary.json`, copy `metrics.png`, `summary.json`, and the sample images into this repository's `results/` folder. Update the README's project row with the measured test metrics and dataset counts. Keep `best.pt` locally or publish it as a release asset; model weights are excluded from Git by `.gitignore`. Do not claim a metric unless it appears in this run's `summary.json`.